<a href="https://colab.research.google.com/github/pilaaaar/ANALISIS-NUMERICO/blob/main/LOGISTIC_REGRESSION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

# Cargar el archivo CSV en un DataFrame de pandas
df = pd.read_csv('/content/DM.csv')

# Mostrar las primeras 5 filas del DataFrame para tener una idea de los datos
print("Primeras 5 filas del DataFrame:")
display(df.head())

# Mostrar información general del DataFrame, incluyendo tipos de datos y valores no nulos
print("\nInformación general del DataFrame:")
df.info()

Primeras 5 filas del DataFrame:


,Grupo,Participante,Minuto,C3,C4,CZ,EMG,F3,F4,F7,...,O1,O2,P3,P4,PZ,ROG,T3,T4,T5,T6
0,0,EMV,1,1.457191,1.465362,1.389850,1.469966,1.471585,1.383370,1.416681,...,1.419505,1.460171,1.435626,1.401177,1.405440,1.190718,1.486129,1.455536,1.457664,1.464154
1,0,EMV,10,1.442564,1.372664,1.361794,1.310307,1.477391,1.366440,1.439025,...,1.417780,1.421081,1.410483,1.395753,1.393475,1.116700,1.474367,1.413287,1.477452,1.412524
2,0,EMV,11,1.445738,1.397893,1.363018,1.246467,1.451693,1.372738,1.398308,...,1.444138,1.434578,1.425999,1.411785,1.403913,1.118753,1.472099,1.433305,1.484939,1.440299
3,0,EMV,12,1.381500,1.422462,1.443394,1.445102,1.422085,1.413301,1.450779,...,1.427840,1.451817,1.403597,1.434047,1.428170,1.167026,1.428117,1.453502,1.373695,1.437340
4,0,EMV,13,1.443665,1.355041,1.360214,1.301061,1.474152,1.365256,1.433039,...,1.415445,1.402281,1.391482,1.395755,1.400138,1.139237,1.464603,1.404249,1.453814,1.405438



Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 25 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Grupo         900 non-null    int64  
 1   Participante  900 non-null    object 
 2   Minuto        900 non-null    int64  
 3   C3            900 non-null    float64
 4   C4            900 non-null    float64
 5   CZ            900 non-null    float64
 6   EMG           450 non-null    float64
 7   F3            900 non-null    float64
 8   F4            900 non-null    float64
 9   F7            900 non-null    float64
 10  F8            900 non-null    float64
 11  FP1           900 non-null    float64
 12  FP2           900 non-null    float64
 13  FZ            900 non-null    float64
 14  LOG           900 non-null    float64
 15  O1            900 non-null    float64
 16  O2            900 non-null    float64
 17  P3            900 non-null    float64

In [4]:
# 1. Codificación 'One-Hot' para la columna 'Participante' - Eliminamos esta línea ya que vamos a quitar la columna.
# df_encoded = pd.get_dummies(df, columns=['Participante'], drop_first=True) # Ya no necesitamos codificarla si la eliminamos.

# 2. Definir las características (X) y la variable objetivo (y)
# Ignoramos y eliminamos la columna 'EMG' Y'Participante' para evitar fuga de datos.
X = df.drop(columns=['Grupo', 'Minuto', 'EMG', 'Participante'])
y = df['Grupo']

print("Dimensiones de las características (X):", X.shape)
print("Dimensiones de la variable objetivo (y):", y.shape)
print("\nPrimeras filas de las características (X) después de la modificación:")
display(X.head())

Dimensiones de las características (X): (900, 21)
Dimensiones de la variable objetivo (y): (900,)

Primeras filas de las características (X) después de la modificación:


,C3,C4,CZ,F3,F4,F7,F8,FP1,FP2,FZ,...,O1,O2,P3,P4,PZ,ROG,T3,T4,T5,T6
0,1.457191,1.465362,1.389850,1.471585,1.383370,1.416681,1.335028,1.172173,1.166584,1.348589,...,1.419505,1.460171,1.435626,1.401177,1.405440,1.190718,1.486129,1.455536,1.457664,1.464154
1,1.442564,1.372664,1.361794,1.477391,1.366440,1.439025,1.269399,1.220473,1.181741,1.312856,...,1.417780,1.421081,1.410483,1.395753,1.393475,1.116700,1.474367,1.413287,1.477452,1.412524
2,1.445738,1.397893,1.363018,1.451693,1.372738,1.398308,1.285700,1.211166,1.194989,1.286984,...,1.444138,1.434578,1.425999,1.411785,1.403913,1.118753,1.472099,1.433305,1.484939,1.440299
3,1.381500,1.422462,1.443394,1.422085,1.413301,1.450779,1.404926,1.343141,1.306630,1.423680,...,1.427840,1.451817,1.403597,1.434047,1.428170,1.167026,1.428117,1.453502,1.373695,1.437340
4,1.443665,1.355041,1.360214,1.474152,1.365256,1.433039,1.281785,1.230395,1.214625,1.304981,...,1.415445,1.402281,1.391482,1.395755,1.400138,1.139237,1.464603,1.404249,1.453814,1.405438


In [5]:
# Verificar si cada participante pertenece a un único grupo
participant_group_check = df.groupby('Participante')['Grupo'].nunique()

print("Número de grupos únicos por participante:")
print(participant_group_check)

# Si todos los valores son 1, significa que cada participante pertenece a un solo grupo,
# lo que causaría fuga de datos si se usa 'Participante' como característica.
if (participant_group_check == 1).all():
    print("\nConfirmado: Cada participante está asociado a un único Grupo. La columna 'Participante' está causando fuga de datos.")
else:
    print("\nAtención: Algunos participantes pertenecen a más de un Grupo. La fuga de datos podría deberse a otra razón o a una interacción compleja.")

Número de grupos únicos por participante:
Participante
AEF    1
CLM    1
EMV    1
FGV    1
GH2    1
GUR    1
JAL    1
JAN    1
JGM    1
LIV    1
MGN    1
MJN    1
MMA    1
PCM    1
RAN    1
RLM    1
RRM    1
VCN    1
Name: Grupo, dtype: int64

Confirmado: Cada participante está asociado a un único Grupo. La columna 'Participante' está causando fuga de datos.


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# 3. Dividir los datos en conjuntos de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Dimensiones del conjunto de entrenamiento X_train:", X_train.shape)
print("Dimensiones del conjunto de prueba X_test:", X_test.shape)
print("Dimensiones del conjunto de entrenamiento y_train:", y_train.shape)
print("Dimensiones del conjunto de prueba y_test:", y_test.shape)

# Opcional pero recomendado: Escalado de características numéricas
# La regresión logística se beneficia del escalado de características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Entrenar el modelo de Regresión Logística
model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' es bueno para datasets pequeños y problemas binarios
model.fit(X_train_scaled, y_train)

# 5. Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test_scaled)

# 6. Evaluar el rendimiento del modelo
print("\nPrecisión del modelo (Accuracy):")
print(accuracy_score(y_test, y_pred))

print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

Dimensiones del conjunto de entrenamiento X_train: (720, 21)
Dimensiones del conjunto de prueba X_test: (180, 21)
Dimensiones del conjunto de entrenamiento y_train: (720,)
Dimensiones del conjunto de prueba y_test: (180,)

Precisión del modelo (Accuracy):
0.7888888888888889

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.79      0.85      0.82       100
           1       0.79      0.71      0.75        80

    accuracy                           0.79       180
   macro avg       0.79      0.78      0.78       180
weighted avg       0.79      0.79      0.79       180


Matriz de Confusión:
[[85 15]
 [23 57]]


### Exportar el Reporte de Clasificación a CSV

Para exportar el reporte de clasificación a un archivo CSV, primero debemos convertir el texto del reporte en una estructura de datos que pandas pueda manejar fácilmente, como un DataFrame.

In [7]:
import io
from sklearn.metrics import classification_report # Importar classification_report

# Generar el reporte de clasificación como una cadena de texto
report_str = classification_report(y_test, y_pred, output_dict=True)

# Convertir el diccionario del reporte en un DataFrame
df_report = pd.DataFrame(report_str).transpose()

# Guardar el DataFrame en un archivo CSV
df_report.to_csv('classification_report.csv', index=True)

print("Reporte de clasificación exportado a 'classification_report.csv'")
display(df_report)

Reporte de clasificación exportado a 'classification_report.csv'


,precision,recall,f1-score,support
0,0.787037,0.850000,0.817308,100.000000
1,0.791667,0.712500,0.750000,80.000000
accuracy,0.788889,0.788889,0.788889,0.788889
macro avg,0.789352,0.781250,0.783654,180.000000
weighted avg,0.789095,0.788889,0.787393,180.000000
